# RAG Pipeline - Exp6

* About
  * Able to save LlamaIndex embeddings in Railway and use it in later RAG pipeline
  * Using independent LLM, Embeding models in llamaindex, rather than using the global one
  * Check this RAG pipeline's performance
  * LlamaIndex reference: https://developers.llamaindex.ai/python/framework/integrations/vector_stores/postgres/#create-the-index

* Learnings 🍀
    * `storage_context` made a difference that made LlamaIndex able to upload the embeddings to the pgvector
    * LlamIndex method will add `data_` before your specified table name

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import psycopg2
import pandas as pd

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Before Embedding

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


In [3]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

### Embedding

#### create embeddings --> save in pgvector

* On Local Machine
  * Install DBeaver: https://dbeaver.io/download/
* Railway Setup
  * Railway create a service connect to Github Repo for backend
  * Create --> Template --> Type "pgvector" --> Choose the one with most downloads (psql 18)
    * Don't use the "PgVector" for RAG, that one doesn't have `host` and `password` in public database url, can't upload data from here
  * Connnect Railway service and pgvector by creating a new variable in the service --> `Add Reference` --> `DATABASE_URL`

In [4]:
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")


conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("Vector extension enabled.")

cur.execute("DROP TABLE IF EXISTS rag_dr_embeddings;")
cur.execute("DROP TABLE IF EXISTS data_rag_dr_embeddings_llamindex;")

cur.close()
conn.close()

Connected to database: railway
Vector extension enabled.


In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import VectorStoreIndex
from sqlalchemy import make_url


url = make_url(DATABASE_URL_PUBLIC)
pg_vector_store = PGVectorStore.from_params(
    database=db_name,
    host=url.host,
    password=url.password,
    port=url.port,
    user=url.username,
    table_name="rag_dr_embeddings_llamindex",
    embed_dim=384,
)

embed_model = HuggingFaceEmbedding(
                model_name=all_config['embedding_models'][0], 
                device="cpu",
                embed_batch_size=16
            )

storage_context = StorageContext.from_defaults(vector_store=pg_vector_store)
index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True,  # 🍄
        embed_model=embed_model
    )

print("Embeddings saved to Railway Postgres")

Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Embeddings saved to Railway Postgres


In [10]:
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
cur = conn.cursor()

cur.execute("SELECT current_database();")
print("Connected to database:", cur.fetchone())

cur.execute("""
SELECT table_name FROM information_schema.tables 
WHERE table_schema='public';
""")
print("Tables:", cur.fetchall())

cur.execute("""
SELECT
    relname AS table,
    pg_size_pretty(pg_total_relation_size(relid)) AS size
FROM pg_catalog.pg_statio_user_tables
ORDER BY pg_total_relation_size(relid) DESC;
""")
rows = cur.fetchall()
print("\nTable Sizes:\n")
for row in rows:
    table_name, size = row
    print(f"{table_name:<30} {size}")

cur.close()
conn.close()

Connected to database: ('railway',)
Tables: [('data_rag_dr_embeddings_llamindex',)]

Table Sizes:

data_rag_dr_embeddings_llamindex 408 kB


### RAG After Embedding

In [18]:
import os
import json
import yaml
import asyncio
from concurrent.futures import ProcessPoolExecutor
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

os.environ["GROQ_API_KEY"] = os.environ["GROQ_TOKEN"]
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")

embed_model = HuggingFaceEmbedding(
                model_name=all_config['embedding_models'][0], 
                device="cpu",
                embed_batch_size=16
            )
llm = Groq(model=all_config['llm_models'][0], temperature=0)

vector_store = PGVectorStore.from_params(
    database=db_name,
    host=url.host,
    password=url.password,
    port=url.port,
    user=url.username,
    table_name="data_rag_dr_embeddings_llamindex",
    embed_dim=384,
)

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

Connected to database: railway


In [19]:
FINANCIAL_RAG_SYSTEM_PROMPT = """You are a finance expert.
Your role is to answer financial questions with precision and clarity.

GUIDELINES:
- If data is missing or unclear, state it explicitly - do NOT make up numbers
- Include relevant financial metrics and ratios in your analysis
- Flag any assumptions you make about the data
- For complex queries, structure responses with clear breakdowns

FINANCIAL ACCURACY IS CRITICAL. When in doubt, cite your source and indicate uncertainty.
"""

def get_query_engine(retriever, reranker=None, llm=None):
    node_postprocessors = [reranker] if reranker is not None else []
    return RetrieverQueryEngine.from_args(
        retriever,  # retrieving documents
        node_postprocessors=node_postprocessors,  # a list containing the reranker
        system_prompt=FINANCIAL_RAG_SYSTEM_PROMPT,  # guiding the answer generation
        llm=llm,  # generating the answer
    )

retriever = HybridRetriever(
            index,
            documents,
            top_k=1,
            alpha=0.6,
        )
query_engine = get_query_engine(retriever, reranker=None, llm=llm)
eval_lst = await run_eval_async(rag_lst, query_engine, concurrency=3)
print(len(eval_lst))

30


In [20]:
eval_df = pd.DataFrame(eval_lst)
eval_df.head()

,question,expected_answer,ai_answer,retrieved_lst
0,How to deposit a cheque issued to an associate...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,"[{'metadata': 0, 'content': 'Just have the ass..."
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...","[{'metadata': 1, 'content': 'Sure you can. Yo..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,You can have one Employer Identification Numbe...,"[{'metadata': 1, 'content': 'Sure you can. Yo..."
3,Applying for and receiving business credit,"""I'm afraid the great myth of limited liabilit...",Establishing a relationship with a banker is c...,"[{'metadata': 3, 'content': 'Set up a meeting ..."
4,401k Transfer After Business Closure,You should probably consult an attorney. Howev...,If your employer's 401(k) plan is terminated d...,"[{'metadata': 4, 'content': 'The time horizon ..."


In [ ]:
# def run_llamaindex_rag_pipeline(selected_items, documents, llm_str, embed_model_str,
#                                 vector_index_dir, retriever_params, 
#                                 output_file):
#     llm = Groq(model=llm_str,temperature=0)
#     embed_model = HuggingFaceEmbedding(
#                 model_name=embed_model_str, 
#                 device="cpu",
#                 embed_batch_size=16
#             )
#     storage_context = StorageContext.from_defaults(persist_dir=vector_index_dir)
#     vector_index = load_index_from_storage(storage_context)

#     retriever = HybridRetriever(
#             vector_index,
#             documents,
#             top_k=retriever_params["top_k"],
#             alpha=retriever_params["alpha"]
#         )
#     query_engine = get_query_engine(retriever, reranker=None)

#     eval_lst = asyncio.run(run_eval_async(selected_items, query_engine, concurrency=3))
#     print(len(eval_lst))

#     with open(output_file, "w", encoding="utf-8") as f:
#         json.dump(eval_lst, f, ensure_ascii=False, indent=2)


# def run_one(cfg_path, selected_items, documents):
#     with open(cfg_path, "r", encoding="utf-8") as f:
#         cfg = yaml.safe_load(f)

#     llm_str = cfg["llm_model"]
#     embed_model_str = cfg["embedding_model"]
#     retriever_params = cfg["retriever_params"]
#     output_file = cfg["output_file"]
#     vector_index_dir = cfg["indexing_storage_dir"]

#     return run_llamaindex_rag_pipeline(
#         selected_items,
#         documents,
#         llm_str,
#         embed_model_str,
#         vector_index_dir,
#         retriever_params,
#         output_file,
#     )


# async def run_all_in_processes(cfgs, selected_items, documents, max_workers=2):
#     loop = asyncio.get_running_loop()
#     with ProcessPoolExecutor(max_workers=max_workers) as pool:
#         tasks = [
#             loop.run_in_executor(pool, run_one, cfg_path, selected_items, documents)
#             for cfg_path in cfgs
#         ]
#         await asyncio.gather(*tasks)